# Benchmark Results Plotting

Loads a benchmark results CSV and renders the overhead plots.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('../..'))  # examples/, so `demo` and `spikefi` are importable

from IPython.display import display
import pandas as pd

from demo.benchmarks.benchmark_a import BenchmarkA
from spikefi.utils.io import make_out_filepath


KIND_ORDER = ['golden', 'golden_spikefi', 'faulty']
FTYPE_ORDER = ['-', 'neuron_hard', 'synapse_hard', 'neuron_param', 'synapse_param']
LAYER_ORDER = ['-', 'SC1', 'SC2', 'SC3', 'SF4a', 'SF4b']

## Benchmark A

In [ ]:
CSV_FNAME = 'benchmark_A_nmnist_cnn_R40_B2500_S1.csv'   # CSV_FNAME = 'benchmark_A_gesture_R40_B66_S1.csv'
CSV_FPATH = make_out_filepath(CSV_FNAME)

bm = BenchmarkA.from_csv(CSV_FPATH)

fig = bm.plot('t_exec', show_err=True, show_gsf=False)
display(fig)

fig = bm.plot('t_setup', show_err=True, show_gsf=False, include_wall_overhead=True)
display(fig)

means = bm.overhead.means.copy()
means.index = pd.MultiIndex.from_arrays([
    pd.Categorical(means.index.get_level_values('kind'), KIND_ORDER, ordered=True),
    pd.Categorical(means.index.get_level_values('ftype'), FTYPE_ORDER, ordered=True),
    pd.Categorical(means.index.get_level_values('layer'), LAYER_ORDER, ordered=True),
], names=means.index.names)
means[['t_setup', 't_setup_wall']] *= 1000  # s -> ms, for display only
means = means.rename(columns={'t_setup': 't_setup (ms)', 't_setup_wall': 't_setup_wall (ms)', 't_exec': 't_exec (s)'})
means = means.round({'t_setup (ms)': 3, 't_setup_wall (ms)': 3, 't_exec (s)': 6})
means.sort_index()